In [ ]:
import gym
import matplotlib.pyplot as plt
import numpy as np
from torch import nn

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

In [ ]:
import os
from pathlib import Path

# Change to project root (parent of this notebooks/ folder) — works on any OS
os.chdir(Path(os.path.abspath("")).resolve().parent)
print("Working directory:", os.getcwd())

In [ ]:
import sys
sys.path.append("../")

In [ ]:
#from mbt_gym.agents.BaselineAgents import CarteaJaimungalMmAgent
from mbt_gym.gym.helpers.N_generate_trajectory import generate_trajectory  # Multi-asset
from mbt_gym.gym.N_StableBaselinesTradingEnvironment import StableBaselinesTradingEnvironment  # Multi-asset wrapper
from mbt_gym.gym.N_Bek_TradingEnvironment import Bek_TradingEnvironment  # Multi-asset env
from mbt_gym.gym.N_wrappers import *  # Multi-asset wrappers

from mbt_gym.rewards.N_RewardFunctions import PnL, CjMmCriterion  # Multi-asset rewards
from mbt_gym.stochastic_processes.N_midprice_models import BrownianMotionMidpriceModel  # Multi-asset
from mbt_gym.stochastic_processes.N_arrival_models import SynchronousHawkesArrivalModel, SynchronousLOBDepthModel  # Multi-asset
from mbt_gym.stochastic_processes.N_fill_probability_models import DynamicLOBExponentialFillFunction  # Multi-asset

from mbt_gym.gym.N_Bek_ModelDynamics import Bek_LimitOrderModelDynamics  # Multi-asset dynamics
from mbt_gym.gym.N_index_names import get_index_map

from mbt_gym.agents.SymmetricPPOPolicy import SymmetricActorCriticPolicy
from environment_set_up.N_env_setup import make_sb_env, make_eval_sb_env
from environment_set_up.N_callbacks import AccurateEvalCallback, SaveAtTimestepsCallback
from environment_set_up.N_run_logging import save_run_metadata
from environment_set_up.N_env_builders import get_cj_env, set_env_globals




In [ ]:
# === GLOBAL CONFIGS ===
terminal_time = 1.0
#arrival_rate = 10.0
#n_steps = int(10 * terminal_time * arrival_rate)
phi = 0.005 #0.01 #0.001
alpha = 0.2 #0.8 # 0.5
n_steps = 100 
num_trajectories = 1000   # or whatever you used to build sb_env
total_timesteps = 100_000_000

# === Seeds ===
network_seed = 789
eval_seed = 456
train_seed = 123

In [ ]:
### <- Prints all relevant index ranges
#env.stochastic_process_indices 

In [ ]:
import numpy as np

# ============================================================
# This checks stability condition for MO arrival intensity processes 
# and LOB depths processes used in Manuscript (Submitted to Futures Market)
# NOTE: This only works when buy and sell side parameters for each asset are the same (symmetric) 

# SIMPLE STABILITY CHECK FOR MO ARRIVAL MODEL (N=2)
#
# Uses theoretical condition:
#
#     (1 - kappa) * beta / (eta + nu + rho) > 1
#
# Interpretation:
#   ratio > 1   -> stable
#   ratio = 1   -> boundary
#   ratio < 1   -> unstable
#
# Returns:
#   stable            : True / False
#   stability_ratio   : minimum ratio across assets
#   lhs               : (1-kappa)*beta
#   rhs               : eta + nu + rho
#   ratios            : per-asset ratios
#   rho_eff           : largest cross-asset influence
# ============================================================

def check_mo_stability(
    mean_reversion_speed,
    self_jump_size,
    mutual_jump_size,
    synchrony_factor,
    cross_asset_influence,
):
    beta = np.asarray(mean_reversion_speed, dtype=float)
    eta = np.asarray(self_jump_size, dtype=float)
    nu = np.asarray(mutual_jump_size, dtype=float)
    kappa = np.asarray(synchrony_factor, dtype=float)
    rho_mat = np.asarray(cross_asset_influence, dtype=float)

    # Effective rho = largest off-diagonal cross-asset term
    rho_eff = np.max(np.abs(rho_mat - np.diag(np.diag(rho_mat))))

    # Compute LHS and RHS
    lhs = (1.0 - kappa) * beta
    rhs = eta + nu + rho_eff

    # Stability ratios
    with np.errstate(divide="ignore", invalid="ignore"):
        ratios = lhs / rhs

    stable = np.all(ratios > 1.0)
    stability_ratio = np.min(ratios)

    return {
        "stable": bool(stable),
        "stability_ratio": float(stability_ratio),
        "lhs": lhs,
        "rhs": rhs,
        "ratios": ratios,
        "rho_eff": float(rho_eff),
    }


# ============================================================
# FIXED PARAMETERS (MO ARRIVAL MODEL)
# ============================================================

mean_reversion_speed = np.array([15.0, 18.0])
self_jump_size       = np.array([3.0, 3.2])
mutual_jump_size     = np.array([1.5, 1.6])
synchrony_factor     = np.array([0.05, 0.05])


# ============================================================
# FIXED PARAMETERS (LOB DEPTH MODEL)
# ============================================================

c_mean_reversion_speed = np.array([10.0, 12.0])
c_self_jump_size       = np.array([0.6, 0.65])
c_mutual_jump_size     = np.array([0.3, 0.32])
c_synchrony_factor     = np.array([0.05, 0.05])


# ============================================================
# SPECIFIC VALUES TO TEST
# ============================================================

X_values = [1, 3, 5, 7, 9]   # MO cross-asset influence
Y_values = [1, 2, 3, 4, 5]   # LOB cross-asset influence


# ============================================================
# MO STABILITY CHECK
# ============================================================

print("\n===== MO ARRIVAL MODEL STABILITY =====")

for X in X_values:

    cross_asset_influence = np.array([
        [0.0, X],
        [X, 0.0],
    ], dtype=float)

    mo_result = check_mo_stability(
        mean_reversion_speed=mean_reversion_speed,
        self_jump_size=self_jump_size,
        mutual_jump_size=mutual_jump_size,
        synchrony_factor=synchrony_factor,
        cross_asset_influence=cross_asset_influence,
    )

    print("\n----------------------------------------")
    print("X =", X)
    print("stable =", mo_result["stable"])
    print("stability_ratio =", mo_result["stability_ratio"])
    print("rho_eff =", mo_result["rho_eff"])
    print("lhs =", mo_result["lhs"])
    print("rhs =", mo_result["rhs"])
    print("ratios =", mo_result["ratios"])


# ============================================================
# LOB DEPTH STABILITY CHECK
# ============================================================

print("\n\n===== LOB DEPTH MODEL STABILITY =====")

for Y in Y_values:

    c_cross_asset_influence = np.array([
        [0.0, Y],
        [Y, 0.0],
    ], dtype=float)

    lob_result = check_mo_stability(
        mean_reversion_speed=c_mean_reversion_speed,
        self_jump_size=c_self_jump_size,
        mutual_jump_size=c_mutual_jump_size,
        synchrony_factor=c_synchrony_factor,
        cross_asset_influence=c_cross_asset_influence,
    )

    print("\n----------------------------------------")
    print("Y =", Y)
    print("stable =", lob_result["stable"])
    print("stability_ratio =", lob_result["stability_ratio"])
    print("rho_eff =", lob_result["rho_eff"])
    print("lhs =", lob_result["lhs"])
    print("rhs =", lob_result["rhs"])
    print("ratios =", lob_result["ratios"])

In [ ]:
import numpy as np

# This checks stability condition for MO arrival intensity processes 
# and LOB depths processes used in Manuscript (Submitted to Futures Market)
# NOTE:  Works when buy and sell side parameters for each asset are the same (symmetric) or Assymetric 
# Works for both cases. 

# ============================================================
# PARAMETERS (MO ARRIVAL MODEL)
# ============================================================

mean_reversion_speed = np.array([15.0, 18.0])   # beta
self_jump_size       = np.array([3.0, 3.2])     # eta
mutual_jump_size     = np.array([1.5, 1.6])     # nu
synchrony_factor     = np.array([0.05, 0.05])   # kappa

# ============================================================
# ASYMMETRIC CROSS-ASSET INFLUENCE
# ============================================================

cross_asset_influence = np.array([
    [0.0, 3.0],
    [3.0, 0.0],
])

# ============================================================
# BUILD INTERACTION MATRIX
# ============================================================

# A = excitation matrix
A = np.array([
    [self_jump_size[0] + mutual_jump_size[0], cross_asset_influence[0, 1]],
    [cross_asset_influence[1, 0], self_jump_size[1] + mutual_jump_size[1]],
])

# D = mean-reversion scaling (diagonal)
D = np.diag((1.0 - synchrony_factor) * mean_reversion_speed)

# Effective system matrix
M = np.linalg.inv(D) @ A

# ============================================================
# SPECTRAL RADIUS
# ============================================================

eigvals = np.linalg.eigvals(M)
spectral_radius = np.max(np.abs(eigvals))

stable = spectral_radius < 1.0

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n===== ASYMMETRIC STABILITY CHECK =====")
print("Cross-asset influence matrix:\n", cross_asset_influence)

print("\nMatrix A (excitation):\n", A)
print("\nMatrix D (mean reversion scaling):\n", D)
print("\nEffective matrix M = D^{-1} A:\n", M)

print("\nEigenvalues:", eigvals)
print("Spectral radius:", spectral_radius)

print("\nStable:", stable)

In [ ]:
# choose num_assets
num_assets = 2

do_reward_scaling = False          # <- flip to True when you want scaling
baseline_fixed_depth = 0.5         # <- change this whenever you want (only matters if scaling=True)

# === 2-asset BASE MODEL (Trial1) dynamics == builder defaults (from saved metadata) ===
# cross-asset influence = 0.05 for BOTH MO intensities and LOB depths
cross_asset_influence = [[0.0, 0.05],
                         [0.05, 0.0]]
c_cross_asset_influence = [[0.0, 0.05],
                           [0.05, 0.0]]

train_bundle = make_sb_env(
    get_env_fn=get_cj_env,
    num_trajectories=num_trajectories,
    #baseline_num_total_trajectories=100_000,
    num_assets=num_assets,
    do_reward_scaling=do_reward_scaling,
    baseline_fixed_depth=baseline_fixed_depth,  # <- only matters if scaling=True
    env_seed=train_seed,
    debug=True,
    cross_asset_influence=cross_asset_influence,     # Trial1: [[0,0.05],[0.05,0]]
    c_cross_asset_influence=c_cross_asset_influence, # Trial1: [[0,0.05],[0.05,0]]
)


eval_bundle = make_eval_sb_env(
    get_env_fn=get_cj_env,
    train_bundle=train_bundle,
    eval_seed=eval_seed,
    debug=True,
    cross_asset_influence=cross_asset_influence,
    c_cross_asset_influence=c_cross_asset_influence,
)

sb_env = train_bundle.sb_env
sb_eval_env = eval_bundle.sb_eval_env
state_indices = train_bundle.state_indices


In [ ]:
# VERY Important
# Do NOT forget to change trial_num
#
# !!! WARNING: trial_num = 1 is the ORIGINAL 2-asset BASE MODEL (PPO_2Assets_Trial1).
#     Re-running training with trial_num = 1 will OVERWRITE the saved base-model
#     checkpoints in N_SB_models/PPO_Checkpoints_2Assets/PPO_2Assets_Trial1/.
#     Change trial_num to a NEW number before running unless you intend to reproduce it.

import os

sb_env = VecMonitor(sb_env)
sb_eval_env = VecMonitor(sb_eval_env)

trial_num = 1   # 2-asset BASE MODEL (original truly-coupled case)

run_name = f"PPO_{num_assets}Assets_Trial{trial_num}"
#tag = "SingleAsset" if num_assets == 1 else "MultiAsset"
tag = "SingleAsset" if num_assets == 1 else f"{num_assets}Assets"


tensorboard_logdir = f"./N_tensorboard/PPO_{tag}/{run_name}/"

# IMPORTANT: EvalCallback expects this to be a DIRECTORY
best_model_path = f"./N_SB_models/PPO_Best_{tag}/{run_name}"

# Checkpoint directory
ckpt_root = f"./N_SB_models/PPO_Checkpoints_{tag}/{run_name}"

# Create directories
os.makedirs(tensorboard_logdir, exist_ok=True)
os.makedirs(best_model_path, exist_ok=True)
os.makedirs(ckpt_root, exist_ok=True)


In [ ]:
# with lr schedular

# === Seeds ===
network_seed = 789
eval_seed = 456

# === Learning rate scheduler ===
def lr_scheduler(progress: float) -> float:
    reversed_progress = 1 - progress
    initial_lr = 0.0003
    final_lr = 0.00003
    return initial_lr - (initial_lr - final_lr) * reversed_progress

# === Policy architecture ===
policy_kwargs = dict(
    net_arch=[dict(pi=[256, 256, 256], vf=[256, 256, 256])]
)

# === PPO parameters ===
PPO_params = {
    "policy":  SymmetricActorCriticPolicy, #"MlpPolicy",
    "env": sb_env,
    "verbose": 1,
    "policy_kwargs": policy_kwargs,
    "tensorboard_log": tensorboard_logdir,
    "n_epochs": 3,
    "batch_size": int(n_steps * num_trajectories / 10),
    "n_steps": int(n_steps),
    "learning_rate": lr_scheduler,
    "ent_coef": 0.0,
    "seed": network_seed,
     "gamma": 1.0,      # turn off discounting
}

# === Eval callback params (best model saver) ===
callback_params = dict(
    #eval_env=sb_env,
    eval_env=sb_eval_env,  # use separate eval env
    n_eval_episodes=1000,
    best_model_save_path=best_model_path,  # directory
    deterministic=True,
    eval_freq=500,
    verbose=2,
)

best_cb = AccurateEvalCallback(**callback_params, seed=eval_seed)

# === Checkpoint callback instance ===
#save_steps = [30_000_000, 35_000_000, 40_000_000, 45_000_000, 50_000_000, 55_000_000, 60_000_000, 65_000_000
              #, 70_000_000, 75_000_000, 80_000_000, 85_000_000, 90_000_000, 95_000_000, 100_000_000 ]
save_steps = [30_000_000, 50_000_000, 70_000_000, 100_000_000]

save_cb = SaveAtTimestepsCallback(
    save_steps=save_steps,
    save_dir=ckpt_root,
    name_prefix=run_name,
    verbose=1,
)

# === Create model ===
model = PPO(**PPO_params, device="cpu")


In [ ]:
baseline_mean_reward_to_log = train_bundle.baseline_mean if do_reward_scaling else None
baseline_fixed_depth_to_log = baseline_fixed_depth if do_reward_scaling else None

run_meta, paths = save_run_metadata(
    run_name=run_name,
    tag=tag,
    total_timesteps=total_timesteps,
    num_assets=num_assets,
    num_trajectories=num_trajectories,
    n_steps=n_steps,
    terminal_time=terminal_time,
    phi=phi,
    alpha=alpha,
    state_indices=state_indices,
    train_bundle=train_bundle,
    model=model,
    PPO_params=PPO_params,
    policy_kwargs=policy_kwargs,
    best_cb=best_cb,
    callback_params=callback_params,
    train_seed=train_seed,
    network_seed=network_seed,
    eval_seed=eval_seed,
    baseline_mean_reward=baseline_mean_reward_to_log,      # None if not scaled
    baseline_fixed_depth=baseline_fixed_depth_to_log,      # None if not scaled
)


In [ ]:

# Actual training happens 
import time

start_time = time.time()

#total_timesteps = 15_000_000 # this was defined earlier 
print(f"Training PPO model for {total_timesteps:,} timesteps...")

model.learn(
    total_timesteps=total_timesteps,
    callback=[save_cb, best_cb]  # both callbacks
)

end_time = time.time()
total_time = end_time - start_time
hours, remainder = divmod(total_time, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"Total training time: {int(hours)}h {int(minutes)}m {int(seconds)}s")

print(f"Best mean reward achieved: {best_cb.best_mean_reward}")
print(f"Best model saved at step: {best_cb.best_model_step}")
print(f"Number of evals performed: {best_cb.actual_eval_calls}")

In [ ]:
print(type(model.policy))
print(hasattr(model.policy, "_unpack_actions_flat"))
print(hasattr(model.policy, "_symmetrize_obs_for_asset"))


In [ ]:
import os
os.getcwd()